# Cache-first ELI teleconnection metrics

This notebook diagnoses lead-dependent teleconnections between the Equatorial Longitude Index (ELI) and one downstream gridded variable. ELI is the longitude centroid of the equatorial Pacific warm pool, not a rectangular regional-mean SST index. It shares preparation contracts and reusable processors with:

- `8a_refactor_eli_skill_ts.ipynb`: ELI definitions and lead-time skill analysis;
- `3b_refactor_sst_telecon.ipynb`: the corresponding regional SST-index teleconnection workflow;
- `1a_refactor_atm_leadtime_acc_skill_map.ipynb`: atmospheric prepared anomaly bundles and prepared observations;
- `1b_refactor_lnd_leadtime_acc_skill_map.ipynb`: prepared land forecast and reference bundles.

Those notebooks are not execution prerequisites: `inputs.mode="auto"` prepares only the selected products that are unavailable.

**Input policy:** by default this notebook reuses compatible analysis-ready products and prepares only missing or incompatible selected inputs. Set `CONFIG["inputs"]["mode"]` to `"require"` for an archive-free cache-only run or `"rebuild"` to regenerate the selected dependencies.

For each system, initialization month, and lead, the notebook estimates:

1. the observed teleconnection map: correlation of the observed HadISST2 ELI anomaly with the observed downstream anomaly;
2. the forecast teleconnection map: correlation of the ensemble-mean E3SM ELI anomaly with the ensemble-mean downstream anomaly across initialization years;
3. scalar fidelity metrics comparing forecast and observed maps: spatial pattern correlation, centered spatial RMSE, regression slope/amplitude ratio, sign agreement, significant-area fractions, and sample counts.

Correlation maps, p-values, and scalar metrics are saved to a provenance-rich NetCDF file. A lead-summary figure and selected map panels are also saved.

In [ ]:
import os
import sys
from pathlib import Path

_repo_override = os.environ.get("ESP_LAB_REPO_ROOT")
_repo_candidates = (
    [Path(_repo_override).expanduser().resolve()]
    if _repo_override
    else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
)
REPO_ROOT = next((p for p in _repo_candidates if (p / "workflows" / "diagnostics").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside ESP-Lab or set ESP_LAB_REPO_ROOT to its checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import hashlib
import importlib
import json
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAVE_CARTOPY = True
except ImportError:
    HAVE_CARTOPY = False

try:
    from scipy import stats as scipy_stats
except ImportError as exc:
    raise ImportError("This notebook requires scipy for p-values") from exc

from esp_lab.leadtime_plot_utils import seasonal_label
from esp_lab.utils import colormap_utils as mycolors
from workflows.diagnostics import sst_teleconnections as telecon
telecon = importlib.reload(telecon)  # pick up local workflow edits in a live kernel

from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

%matplotlib inline


## Managed Dask resources

A local production run uses worker processes with the shared-node safety cap.
Rerunning this cell closes any previous notebook cluster and tracked datasets first.


In [ ]:
import dask
from dask.distributed import (
    wait,
    get_client,
)
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

dask.__version__

# User-adjustable Dask settings (shared login nodes are capped at four workers).
DASK_SETTINGS = {
    "enabled": True,
    "cluster_type": "local",
    "workers": 12,
    "memory_limit": "4GB",
}

dask_cfg = DaskConfig(
    cluster_type=DASK_SETTINGS["cluster_type"],
    workers=DASK_SETTINGS["workers"],
    memory_limit=DASK_SETTINGS["memory_limit"],
)
cluster, client, workflow_resources = restart_notebook_cluster(
    globals(),
    lambda: get_cluster_client(dask_cfg) if DASK_SETTINGS["enabled"] else (None, None),
)
if client is not None:
    display(client)


## 1. User configuration

Normally only `tel_var`, `inputs`, and `regrid` need editing. This workflow fixes the upstream index to ELI. `inputs.eli_grid` selects regridded E3SM SST (`regridded`) or native MPAS-Ocean SST (`native`) for the model ELI; the observed ELI is computed from HadISST2. The seasonal contract uses centered three-month means.

In [ ]:
# Explicit teleconnection pair selected for this run.
upstream_index = "ELI"
tel_var = "PRECT" #"TS" #"H2OSOI" #"TREFHT"

# Optional common processing end year. None uses the end year from
# selection.initialization_years.
YEAR_END = 2011  # None
CONFIG = {
    "dask": {
        # Lazy cache reads; computation uses the managed scheduler above.
        "chunks": "auto",
        "persist_metrics": True,
    },
    "paths": {
        "diag_root": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag",
        "output_dir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/teleconnections",
        "figure_dir": "/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/teleconnections",
    },
    "inputs": {
        "mode": "auto",  # auto | require | rebuild
        "initialization_years": (1980, 2018 if YEAR_END is None else int(YEAR_END)),
        "year_end": YEAR_END,
        "raw_model_root": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
        "observation_root": "/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series",
        "ensemble_member_count": 10,
        "monthly_nlead": 24,
        "workers": 4,
        "sst_land_mask": True,
        "eli_grid": "regridded",  # regridded | native
        # For native ELI, optionally set eli_mesh_file and eli_raw_model_root.
    },
    "selection": {
        "upstream_index": upstream_index,
        "downstream_variable": tel_var,
        "systems": ["E3SM-FOSIRL", "E3SM-Reanalysis", "E3SM-4DEnVarOcn"],
        "init_months": [5, 11],
        "leads": "all",  # all common complete seasonal leads, or a list of stored L values
        "year_end": YEAR_END,
        "verification_years": (1981, 2018 if YEAR_END is None else int(YEAR_END)),
        "climatology_years": (1981, 2010),
    },
    # Desired grid contract for either compatible-cache reuse or on-demand preparation.
    "regrid": {
        "target_dlat": 5.0,
        "target_dlon": 5.0,
        "method": "conservative",
        "periodic": True,
    },
    "analysis": {
        "detrend": True,
        "alpha": 0.10, # significance level
        "minimum_years": 20,
        "latitude_bounds": (-80.0, 80.0),
        "area_weighted": True,
        "sign_agreement_threshold": 0.0,
    },
    "cache": {
        "mode": "auto",  # auto | rebuild | require
        "allow_ambiguous_matches": True,
    },
    "figures": {
        "map_leads": None,  # Stored L coordinates; None = every available seasonal lead
        "dpi": 300,
        "cmap": "blue2red_acc",
        "correlation_limits": (-1.0, 1.0),
        "correlation_interval": 0.1,
        "correlation_cutoff": 0.5,
        "colorbar_ticks": (-0.9, -0.6, -0.3, 0.0, 0.3, 0.6, 0.9),
        "show_significant_only": True,
    },
}

E3SM_CASES = telecon.E3SM_CASES
DOWNSTREAM_VARIABLES = telecon.DOWNSTREAM_VARIABLES

index_name = CONFIG["selection"]["upstream_index"]
if index_name != "ELI":
    raise ValueError(f"This notebook requires upstream_index='ELI', got {index_name!r}")
if CONFIG["inputs"].get("eli_grid", "regridded") not in {"regridded", "native"}:
    raise ValueError("inputs.eli_grid must be regridded or native")
if tel_var not in DOWNSTREAM_VARIABLES:
    raise ValueError(f"Unconfigured downstream variable {tel_var!r}; choose from {list(DOWNSTREAM_VARIABLES)}")
if CONFIG["inputs"]["mode"] not in {"auto", "rebuild", "require"}:
    raise ValueError("inputs.mode must be auto, rebuild, or require")
if CONFIG["cache"]["mode"] not in {"auto", "rebuild", "require"}:
    raise ValueError("cache.mode must be auto, rebuild, or require")
_sel_verif_y1 = CONFIG["selection"]["verification_years"][1]


## 2. Cache registry and discovery

This registry separates scientific names from physical files. Atmospheric prepared bundles contain `anomaly`, `climatology`, and `time`; land bundles contain the configured field plus `time`. Observation caches contain `observation` (atmosphere) or the land reference variable.

Hashed/versioned products are selected by metadata, not merely by filename. If more than one compatible candidate remains, discovery stops unless the user explicitly permits choosing the newest match.

In [ ]:
DIAG_ROOT = Path(CONFIG["paths"]["diag_root"])
OUTPUT_DIR = Path(CONFIG["paths"]["output_dir"])
FIGURE_DIR = Path(CONFIG["paths"]["figure_dir"])

# Expose reusable workflow helpers
select_cache = telecon.select_cache
upstream_paths = telecon.upstream_sst_paths
resolve_downstream_paths = telecon.resolve_downstream_paths

# Reuse or prepare selected upstream products, then validate the inventory.
inventory = telecon.ensure_upstream_products(CONFIG)
display(inventory)

missing = inventory.query("status == 'missing'")
if not missing.empty:
    raise FileNotFoundError(
        "Required upstream products remain missing after applying inputs.mode.\n"
        + missing.to_string(index=False)
    )
skipped = inventory.query("status == 'skipped'")
if not skipped.empty:
    print(f"Note: {len(skipped)} combination(s) skipped (e.g. system has no land component).")


## 3. Alignment and anomaly helpers

Alignment is by **target year derived from each cache's valid-time coordinate**, never by positional index. This matters because initialization year and verification year differ at long leads. Forecast SST and downstream fields must share the same initialization-year/lead grid and valid target season.

The 1a `anomaly` field is already lead-dependent drift corrected. Land caches are normalized here only when they are not explicitly marked as anomalies. Observed fields and indices are converted to monthly-climatology anomalies over the configured climatology window, then sampled at the forecast target dates.

In [ ]:
time_year_month = telecon.time_year_month
parse_init_years = telecon.parse_init_years
lead_signature = telecon.lead_signature
match_leads = telecon.match_leads
linear_detrend = telecon.linear_detrend
monthly_anomaly = telecon.monthly_anomaly
lead_anomaly = telecon.lead_anomaly
observed_at_valid_time = telecon.observed_at_valid_time
corr_and_p = telecon.corr_and_p
weighted_spatial_metrics = telecon.weighted_spatial_metrics


## 4. Teleconnection computation

The scalar comparison domain is the intersection of finite forecast/observed maps and the configured latitude band. Cosine-latitude weights are used by default. The forecast map is based on the ensemble mean for both ELI and the downstream field; this measures the predictable, forced teleconnection rather than within-ensemble weather covariance.

The output key includes all scientific choices. `auto` reuses an exact compatible metrics file; `rebuild` replaces it; `require` refuses to compute if it is absent. Source paths, modification times, and sizes enter the fingerprint so upstream cache changes invalidate downstream teleconnection metrics.

In [ ]:
def compute_one(system, init_month, variable):
    """Compute lead-dependent teleconnection metrics for one (system, init_month, variable) case."""
    return telecon.compute_system_teleconnection(
        system, init_month, variable, CONFIG,
        resource_tracker=workflow_resources,
    )

metrics_ds, out_file, cache_status = telecon.ensure_teleconnection_dataset(
    CONFIG, inventory=inventory, resource_tracker=workflow_resources
)
fingerprint = metrics_ds.attrs.get("fingerprint", "latest")
print(f"Teleconnection metrics ({cache_status}): {out_file}")
display(metrics_ds)


## 6. Analysis figures

- The `map panels` are generated first, placing the observed reference beside every forecast system for each initialization month. The fidelity summary follows, with each initialization month in a separate row. When `show_significant_only` is enabled, grid cells that do not meet the configured pointwise significance level are masked; interpret this as descriptive unless a field-significance/FDR extension is added.
- `pattern_correlation` asks whether the forecast reproduces the geographical shape of the observed teleconnection.
- `centered_rmse` measures spatial pattern error after removing each map's area-weighted mean.
- `amplitude_ratio` and `regression_slope` distinguish weak/strong patterns and sign reversal.
- `sign_agreement_fraction` is an intuitive spatial consistency measure.
- Pointwise p-values are saved, but multiple-testing control and field significance are not claimed.

In [ ]:
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.offsetbox import AnchoredText

def initialization_label(month):
    return pd.Timestamp(2000, int(month), 1).strftime("%b").upper()


def display_lead(lead):
    """Convert the stored centered-season L coordinate to 1a's displayed lead."""
    return int(lead) - 2

def draw_map(ax, da, *, pvalue=None):
    lat = "lat" if "lat" in da.coords else "latitude"
    lon = "lon" if "lon" in da.coords else "longitude"
    lower, upper = correlation_limits
    interval = correlation_interval
    levels = np.arange(lower, upper + 0.5 * interval, interval)
    cmap = (
        mycolors.blue2red_acc_cmap(levels, correlation_cutoff)
        if cmap_name == "blue2red_acc" else plt.get_cmap(cmap_name)
    )
    norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)
    kwargs = dict(
        x=lon, y=lat, ax=ax, cmap=cmap, norm=norm,
        add_colorbar=False, transform=ccrs.PlateCarree() if HAVE_CARTOPY else None
    )
    image = da.plot(**{k: v for k, v in kwargs.items() if v is not None})

    # Overlay significance test as dot hatch instead of masking
    if show_significant and pvalue is not None:
        sig = (pvalue.values <= alpha_sig).astype(float)
        if np.any(sig > 0):
            ax.contourf(
                da[lon].values, da[lat].values, sig,
                levels=[0.5, 1.5],
                colors="none",
                hatches=[sig_hatch_pattern],
                transform=ccrs.PlateCarree() if HAVE_CARTOPY else None,
            )

    if HAVE_CARTOPY:
        ax.set_global()
        ax.coastlines(linewidth=coastline_linewidth, color=coastline_color)
        ax.add_feature(cfeature.BORDERS, linewidth=border_linewidth, edgecolor=border_edgecolor)
        ax.gridlines(linewidth=gridline_linewidth, color=gridline_color, alpha=gridline_alpha, linestyle=gridline_linestyle)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")  # suppress xarray's scalar-coordinate title
    return image


In [ ]:
# =========================================================================
# 6. Multi-Panel Teleconnection Correlation Maps - Setup Parameters
# =========================================================================
# --- Typography & Base Font ---
fontz = 14
fs_suptitle = fontz * 1.0       # 17 pt bold
fs_group_header = fontz * 0.95  # 15 pt bold
fs_col_title = fontz * 0.95     # 12 pt bold
fs_cbar_label = fontz * 0.9     # 13 pt bold
fs_cbar_ticks = fontz * 0.85    # 11 pt
fs_axis_ticks = fontz * 0.8     # 9.5 pt
fs_badge = fontz * 0.8          # 9.5 pt bold
fs_unavailable = fontz * 0.85   # 10 pt

# --- Subplot Lead Badge Styling ---
badge_loc = "lower left"
badge_pad = 0.18
badge_borderpad = 0.25
badge_facecolor = (1.0, 1.0, 1.0, 0.85)
badge_edgecolor = "0.4"
badge_linewidth = 0.5
badge_boxstyle = "round,pad=0.15"

# --- Figure Geometry & GridSpec Layout ---
col_width = 3.3
row_height = 1.75
width_pad = 0.8
height_pad = 2.2
grid_left = 0.035
grid_right = 0.985
grid_bottom = 0.08
grid_top = 0.90
grid_hspace = 0.07
grid_wspace = 0.03
gap_ratio = 0.22                # Gap width between May and Nov groups

# --- Map Features & Ticks ---
lon_ticks = [-120, 0, 120]
lat_ticks = [-60, -30, 0, 30, 60]
coastline_color = "0.25"
coastline_width = 0.5
border_color = "0.45"
border_width = 0.2
gridline_color = "0.65"
gridline_width = 0.25
gridline_alpha = 0.4
gridline_style = ":"

# --- Group Headers & Divider Styling ---
group_header_y = 0.938
group_line_y = 0.925
group_line_color = "0.4"
group_line_width = 1.2
divider_top_y = 0.94
divider_color = "0.65"
divider_style = "--"
divider_width = 1.0
divider_alpha = 0.7
suptitle_y = 0.985

# --- Colorbar Placement & Styling ---
colorbar_rect = [0.18, 0.024, 0.64, 0.016]  # [left, bottom, width, height]
colorbar_ticks = CONFIG["figures"].get("colorbar_ticks", (-0.9, -0.6, -0.3, 0.0, 0.3, 0.6, 0.9))
correlation_limits = CONFIG["figures"].get("correlation_limits", (-1.0, 1.0))
correlation_interval = CONFIG["figures"].get("correlation_interval", 0.1)
correlation_cutoff = CONFIG["figures"].get("correlation_cutoff", 0.5)
cmap_name = CONFIG["figures"].get("cmap", "blue2red_acc")

# --- Significance Overlay ---
show_significant = CONFIG["figures"].get("show_significant_only", True)
alpha_sig = CONFIG["analysis"].get("alpha", 0.10)
sig_hatch = "..."

# --- Selection & Output ---
configured_leads = CONFIG["figures"].get("map_leads", None)
available_leads = [int(lead) for lead in metrics_ds.L.values]
map_leads = (
    available_leads if configured_leads is None
    else [int(lead) for lead in configured_leads if int(lead) in available_leads]
)
plot_systems = [
    system for system in CONFIG["selection"]["systems"]
    if system in set(metrics_ds.system.values.astype(str))
]
plot_months = [
    int(month) for month in CONFIG["selection"]["init_months"]
    if int(month) in set(map(int, metrics_ds.init_month.values))
]
figure_dpi = CONFIG["figures"].get("dpi", 300)

index_display = index_name.replace("Nino", "Niño")

for variable in metrics_ds.variable.values:
    variable_leads = [
        lead for lead in map_leads
        if not metrics_ds.model_correlation.sel(variable=variable, L=lead).isnull().all()
    ]
    if not variable_leads:
        print(f"Skipped {variable}: no model correlation maps are available")
        continue
    nrows = len(variable_leads)
    nsys = len(plot_systems)
    group_width = nsys + 1  # observed reference plus forecast systems
    width_ratios = []
    for month_index in range(len(plot_months)):
        width_ratios.extend([1.0] * group_width)
        if month_index < len(plot_months) - 1:
            width_ratios.append(gap_ratio)
    total_cols = len(width_ratios)
    projection = ccrs.PlateCarree() if HAVE_CARTOPY else None
    fig = plt.figure(figsize=(col_width * len(plot_months) * group_width + width_pad, row_height * nrows + height_pad))
    grid = fig.add_gridspec(
        nrows, total_cols, width_ratios=width_ratios,
        left=grid_left, right=grid_right, bottom=grid_bottom, top=grid_top,
        hspace=grid_hspace, wspace=grid_wspace,
    )
    axes = {}
    image = None
    for month_index, init_month in enumerate(plot_months):
        column_offset = month_index * (group_width + 1)
        for row, lead in enumerate(variable_leads):
            ax = fig.add_subplot(grid[row, column_offset], projection=projection)
            axes[(init_month, "Reference", lead)] = ax
            reference = reference_pvalue = None
            for system in plot_systems:
                candidate = metrics_ds.sel(
                    variable=variable, system=system, init_month=init_month, L=lead
                )
                if not candidate.observed_correlation.isnull().all():
                    reference = candidate.observed_correlation
                    reference_pvalue = candidate.observed_pvalue
                    break
            if reference is None:
                if HAVE_CARTOPY:
                    ax.set_global()
                    ax.coastlines(linewidth=coastline_width, color=coastline_color)
                ax.text(
                    0.5, 0.5, "Unavailable", transform=ax.transAxes,
                    ha="center", va="center", color="0.45", fontsize=fs_unavailable,
                )
                ax.set_xticks([])
                ax.set_yticks([])
            else:
                drawn = draw_map(ax, reference, pvalue=reference_pvalue)
                image = drawn if image is None else image
            if HAVE_CARTOPY:
                ax.set_xticks(lon_ticks, crs=ccrs.PlateCarree())
                ax.set_yticks(lat_ticks, crs=ccrs.PlateCarree())
                ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=False))
                ax.yaxis.set_major_formatter(LatitudeFormatter())
                ax.tick_params(
                    labelsize=fs_axis_ticks, length=2, pad=1,
                    labelbottom=(row == nrows - 1),
                    labelleft=True,
                )
            if row == 0:
                ax.set_title("Observed Reference", fontsize=fs_col_title, fontweight="bold", pad=6)
            season_str = seasonal_label(init_month, display_lead(lead))
            badge_text = f"L{display_lead(lead)}: {season_str}"
            at = AnchoredText(
                badge_text, loc=badge_loc,
                prop=dict(size=fs_badge, weight="bold", family="sans-serif"),
                frameon=True, pad=badge_pad, borderpad=badge_borderpad,
            )
            at.patch.set(
                facecolor=badge_facecolor,
                edgecolor=badge_edgecolor,
                linewidth=badge_linewidth,
                boxstyle=badge_boxstyle,
            )
            ax.add_artist(at)

        for system_index, system in enumerate(plot_systems):
            column = column_offset + 1 + system_index
            for row, lead in enumerate(variable_leads):
                ax = fig.add_subplot(grid[row, column], projection=projection)
                axes[(init_month, system, lead)] = ax
                subset = metrics_ds.sel(
                    variable=variable, system=system, init_month=init_month, L=lead
                )
                if subset.model_correlation.isnull().all():
                    if HAVE_CARTOPY:
                        ax.set_global()
                        ax.coastlines(linewidth=coastline_width, color=coastline_color)
                    ax.text(
                        0.5, 0.5, "Unavailable", transform=ax.transAxes,
                        ha="center", va="center", color="0.45", fontsize=fs_unavailable,
                    )
                    ax.set_xticks([])
                    ax.set_yticks([])
                else:
                    drawn = draw_map(
                        ax, subset.model_correlation, pvalue=subset.model_pvalue
                    )
                    image = drawn if image is None else image
                if HAVE_CARTOPY:
                    ax.set_xticks(lon_ticks, crs=ccrs.PlateCarree())
                    ax.set_yticks(lat_ticks, crs=ccrs.PlateCarree())
                    ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=False))
                    ax.yaxis.set_major_formatter(LatitudeFormatter())
                    ax.tick_params(
                        labelsize=fs_axis_ticks, length=2, pad=1,
                        labelbottom=(row == nrows - 1),
                        labelleft=False,
                    )
                if row == 0:
                    ax.set_title(E3SM_CASES[system]["display_name"], fontsize=fs_col_title, fontweight="bold", pad=6)

    # Group headers (May and Nov)
    for month_index, init_month in enumerate(plot_months):
        left_ax = axes[(init_month, "Reference", variable_leads[0])]
        right_ax = axes[(init_month, plot_systems[-1], variable_leads[0])]
        left_pos = left_ax.get_position()
        right_pos = right_ax.get_position()
        center_x = (left_pos.x0 + right_pos.x1) / 2
        month_name = initialization_label(init_month)
        fig.text(
            center_x, group_header_y,
            f"{month_name.upper()} INITIALIZATION",
            ha="center", va="center",
            fontsize=fs_group_header, fontweight="bold",
            color="0.15",
        )
        fig.add_artist(plt.Line2D(
            [left_pos.x0, right_pos.x1], [group_line_y, group_line_y],
            transform=fig.transFigure,
            color=group_line_color, linewidth=group_line_width,
        ))

    # Vertical divider between the two initialization months
    if len(plot_months) > 1:
        gap_x = (axes[(plot_months[0], plot_systems[-1], variable_leads[0])].get_position().x1 +
                 axes[(plot_months[1], "Reference", variable_leads[0])].get_position().x0) / 2
        bottom_y = axes[(plot_months[0], "Reference", variable_leads[-1])].get_position().y0
        top_y = 0.94
        fig.add_artist(plt.Line2D(
            [gap_x, gap_x], [bottom_y, divider_top_y],
            transform=fig.transFigure,
            color=divider_color, linestyle=divider_style, linewidth=divider_width, alpha=divider_alpha,
        ))

    if image is None:
        plt.close(fig)
        print(f"Skipped {variable}: no model correlation maps are available")
        continue

    significance_note = (
        f"stippled where p ≤ {alpha_sig:.2f}"
        if show_significant else ""
    )
    detrend_note = "linear detrend" if CONFIG["analysis"]["detrend"] else "no detrend"
    notes = ", ".join(filter(None, [detrend_note, significance_note]))
    title_suffix = f"\n({notes})" if notes else ""
    fig.suptitle(
        f"{index_display}–{variable} Teleconnection Correlation Patterns{title_suffix}",
        y=suptitle_y, fontsize=fs_suptitle, fontweight="bold",
    )

    # Colorbar
    colorbar_ax = fig.add_axes(colorbar_rect)
    colorbar = fig.colorbar(
        image, cax=colorbar_ax, orientation="horizontal",
        ticks=colorbar_ticks,
    )
    colorbar.set_label(f"Teleconnection Correlation $r$ ({index_display} vs. {variable})", fontsize=fs_cbar_label, fontweight="bold")
    colorbar.ax.tick_params(labelsize=fs_cbar_ticks, length=3)

    path = (
        FIGURE_DIR / (
            f"teleconnection_{index_name.replace('.', '')}_{variable}_"
            f"correlation_reference_comparison.png"
        )
    )
    fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


In [ ]:
# =========================================================================
# 6. Teleconnection Fidelity Summary - Setup Parameters
# =========================================================================
# --- Typography & Base Font ---
fontz = 14
fs_suptitle = fontz + 2         # 16 pt bold
fs_col_title = fontz            # 14 pt bold
fs_axis_label = fontz - 1       # 13 pt bold
fs_axis_ticks = fontz - 3       # 11 pt
fs_legend = fontz - 2           # 12 pt

# --- Figure Geometry & Grid Layout ---
figsize = (8.5, 10.5)
layout_rect = (0.02, 0.055, 0.98, 0.96)
suptitle_y = 0.985
col_title_pad = 10
lead_pad = 0.8

# --- Line & Marker Appearance ---
series_linewidth = 1.5
series_markersize = 6
grid_color = "0.88"
grid_linewidth = 0.6
grid_linestyle = "-"
grid_alpha = 0.7

# --- Physical Reference Lines ---
ref_amplitude = 1.0
ref_amplitude_style = dict(color="0.55", linestyle="--", linewidth=0.9, alpha=0.7)
ref_correlation = 0.0
ref_correlation_style = dict(color="0.65", linestyle=":", linewidth=0.8, alpha=0.6)
ref_sign_agreement = 0.5
ref_sign_agreement_style = dict(color="0.65", linestyle=":", linewidth=0.8, alpha=0.6)

# --- Legend Settings ---
legend_loc = "lower center"
legend_bbox = (0.5, 0.015)
legend_framealpha = 0.9
legend_edgecolor = "0.8"

# --- Metrics, Labels & Forecast Systems ---
summary_metrics = ["pattern_correlation", "centered_rmse", "amplitude_ratio", "sign_agreement_fraction"]
ylabels = ["Pattern correlation", "Centered RMSE", "Amplitude ratio", "Sign agreement"]
plot_systems = [
    system for system in CONFIG["selection"]["systems"]
    if system in set(metrics_ds.system.values.astype(str))
]
plot_months = [
    int(month) for month in CONFIG["selection"]["init_months"]
    if int(month) in set(map(int, metrics_ds.init_month.values))
]
figure_dpi = CONFIG["figures"].get("dpi", 300)

METHOD_STYLES = {
    "E3SM-4DEnVarOcn": {"short_name": "4DEnVarOcn", "color": "tab:purple", "marker": "o"},
    "E3SM-FOSIRL": {"short_name": "FOSIRL", "color": "black", "marker": "s"},
    "E3SM-Reanalysis": {"short_name": "Reanalysis", "color": "tab:blue", "marker": "D"},
}

index_display = index_name.replace("Nino", "Niño")

for variable in metrics_ds.variable.values:
    valid_leads = [
        int(lead) - 2 for lead in metrics_ds.L.values
        if np.any(np.isfinite(metrics_ds.pattern_correlation.sel(variable=variable, L=lead)))
    ]

    nrows = len(summary_metrics)
    ncols = len(plot_months)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=figsize,
        sharex=True,
        sharey="row",
        squeeze=False,
    )

    for col, init_month in enumerate(plot_months):
        month_label = pd.Timestamp(2000, int(init_month), 1).strftime("%b")
        col_title = f"{month_label} Initialization"

        for row, (metric, ylabel) in enumerate(zip(summary_metrics, ylabels)):
            ax = axes[row, col]

            # Reference lines for physical context
            if metric == "amplitude_ratio":
                ax.axhline(ref_amplitude, **ref_amplitude_style, zorder=1)
            elif metric == "pattern_correlation":
                ax.axhline(ref_correlation, **ref_correlation_style, zorder=1)
            elif metric == "sign_agreement_fraction":
                ax.axhline(ref_sign_agreement, **ref_sign_agreement_style, zorder=1)

            for system in metrics_ds.system.values:
                sys_str = str(system)
                if sys_str not in E3SM_CASES:
                    continue
                series = metrics_ds[metric].sel(
                    variable=variable, system=sys_str, init_month=init_month
                )
                if series.isnull().all():
                    continue

                style = METHOD_STYLES.get(sys_str, {
                    "color": E3SM_CASES[sys_str]["color"],
                    "marker": "o",
                    "short_name": E3SM_CASES[sys_str]["display_name"],
                })

                leads_display = series.L.astype(int) - 2
                ax.plot(
                    leads_display, series,
                    marker=style["marker"],
                    markersize=series_markersize,
                    linewidth=series_linewidth,
                    color=style["color"],
                    label=E3SM_CASES[sys_str]["display_name"],
                    zorder=3,
                )

            ax.grid(True, linestyle=grid_linestyle, color=grid_color, linewidth=grid_linewidth, alpha=grid_alpha)
            ax.tick_params(labelsize=fs_axis_ticks)

            # Column headers on top row
            if row == 0:
                ax.set_title(col_title, fontsize=fs_col_title, fontweight="bold", pad=col_title_pad)

            # Row y-labels on left column
            if col == 0:
                ax.set_ylabel(ylabel, fontsize=fs_axis_label, fontweight="bold")

            # X-axis label and ticks on bottom row
            if row == nrows - 1:
                ax.set_xlabel("Lead time (months)", fontsize=fs_axis_label)
                if valid_leads:
                    ax.set_xticks(valid_leads)
                    ax.set_xlim(valid_leads[0] - lead_pad, valid_leads[-1] + lead_pad)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(
            handles, labels,
            loc=legend_loc,
            ncol=len(handles),
            fontsize=fs_legend,
            frameon=True,
            framealpha=legend_framealpha,
            edgecolor=legend_edgecolor,
            bbox_to_anchor=legend_bbox,
        )

    fig.suptitle(
        f"{index_display}–{variable} Teleconnection Fidelity Summary",
        fontsize=fs_suptitle,
        fontweight="bold",
        y=suptitle_y,
    )
    plt.tight_layout(rect=layout_rect)
    path = FIGURE_DIR / f"teleconnection_{index_name.replace('.', '')}_{variable}_summary.png"
    fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


## 7. Interpretation and recommended extensions

### Taylor diagram synthesis
The Taylor diagram below synthesizes three complementary pattern fidelity metrics simultaneously:
- **Azimuthal angle ($\theta = \arccos(r)$)**: Pattern correlation with the observed teleconnection.
- **Radial distance ($R = \sigma_m / \sigma_o$)**: Spatial amplitude ratio (relative spatial variability).
- **Distance from Reference star ($(R=1.0, \theta=0)$)**: Centered RMS Error (CRMSE).

A perfect forecast teleconnection pattern would coincide with the gold **Reference (Obs)** star at $(1.0, 0^\circ)$.

- `pattern_correlation` asks whether the forecast reproduces the geographical shape of the observed teleconnection.
- `centered_rmse` measures spatial pattern error after removing each map's area-weighted mean.
- `amplitude_ratio` and `regression_slope` distinguish weak/strong patterns and sign reversal.
- `sign_agreement_fraction` is an intuitive spatial consistency measure.
- Pointwise p-values are saved, but multiple-testing control and field significance are not claimed.

Recommended phase-2 additions are paired bootstrap confidence intervals over years, member-wise teleconnection distributions, partial correlations controlling for Niño3.4, lagged index/response seasons, and FDR or field-significance testing. Those should be new configuration options rather than changes to the cache-first input contract.

In [ ]:
# =========================================================================
# 7. Taylor Diagram Synthesis - Setup Parameters
# =========================================================================
# --- Typography & Base Font ---
fontz = 14
fs_suptitle = fontz + 2         # 16 pt bold
fs_panel_title = fontz          # 14 pt bold
fs_axis_label = fontz - 3       # 11 pt
fs_corr_arc = fontz - 4         # 10 pt
fs_ticks = fontz - 4            # 10 pt
fs_legend = fontz - 3           # 11 pt
fs_lead_annot = fontz - 5       # 9 pt bold
fs_crmse_clabel = fontz - 6     # 8 pt

# --- Polar Diagram Geometry ---
r_max = 1.6
r_ticks = [0.5, 1.0, 1.5]
crmse_levels = [0.25, 0.50, 0.75, 1.00, 1.25]
corr_label_offset = 1.13        # Radial distance ratio for correlation arc label

# --- Grid & Contour Aesthetics ---
grid_color = "0.85"
grid_linewidth = 0.6
grid_linestyle = "-"
crmse_contour_color = "0.70"
crmse_contour_style = ":"
crmse_contour_width = 0.7
crmse_contour_alpha = 0.7
crmse_clabel_color = "0.45"

# --- Reference Marker & Standard Deviation Circle ---
ref_circle_color = "0.35"
ref_circle_style = "--"
ref_circle_width = 0.9
ref_circle_alpha = 0.6
ref_marker = "*"
ref_marker_size = 11
ref_marker_color = "gold"
ref_marker_edgecolor = "black"
ref_marker_edgewidth = 0.9

# --- Trajectory Line & Markers ---
connect_leads = True
traj_linewidth = 1.1
traj_line_alpha = 0.35
marker_size_range = (5.0, 10.5)
marker_alpha_range = (0.45, 0.95)

# --- Lead Annotation Box ---
annot_boxstyle = "round,pad=0.12"
annot_facecolor = "white"
annot_alpha = 0.75
annot_xytext = (0, 6)

# --- Subplot & Figure Layout ---
panel_figsize_width_per_month = 7.5
panel_figsize_height = 7.0
panel_title_pad = 24
suptitle_y = 0.98
adjust_bottom = 0.14
adjust_top = 0.86
adjust_wspace = 0.32

# --- Legend Settings ---
legend_loc = "lower center"
legend_bbox = (0.5, 0.01)
legend_framealpha = 0.9
legend_edgecolor = "0.8"

# --- Selection & Systems ---
figure_dpi = CONFIG["figures"].get("dpi", 300)
plot_months = [
    int(month) for month in CONFIG["selection"]["init_months"]
    if int(month) in set(map(int, metrics_ds.init_month.values))
]

METHOD_STYLES = {
    "E3SM-FOSIRL": {"short_name": "FOSIRL", "color": "black", "marker": "s"},
    "E3SM-Reanalysis": {"short_name": "Reanalysis", "color": "tab:blue", "marker": "D"},
    "E3SM-4DEnVarOcn": {"short_name": "4DEnVarOcn", "color": "tab:purple", "marker": "o"},
}

index_display = index_name.replace("Nino", "Niño")

for variable in metrics_ds.variable.values:
    min_corr = float(
        metrics_ds.pattern_correlation.sel(variable=variable).min(skipna=True)
    )
    thetamax = 180 if min_corr < 0 else 90

    if thetamax == 90:
        corr_ticks = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99])
        th_ref = np.linspace(0, np.pi / 2, 100)
        corr_label_angle = np.deg2rad(45)
        corr_label_rotation = -45
    else:
        corr_ticks = np.array([-0.9, -0.7, -0.5, -0.3, -0.1, 0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99])
        th_ref = np.linspace(0, np.pi, 100)
        corr_label_angle = np.deg2rad(90)
        corr_label_rotation = 0

    fig = plt.figure(figsize=(panel_figsize_width_per_month * len(plot_months), panel_figsize_height))
    legend_handles = []
    legend_labels = []

    for idx, init_month in enumerate(plot_months):
        ax = fig.add_subplot(1, len(plot_months), idx + 1, projection="polar")
        ax.set_thetamin(0)
        ax.set_thetamax(thetamax)
        ax.set_ylim(0, r_max)

        # Lighten the background grid substantially
        ax.grid(True, color=grid_color, linewidth=grid_linewidth, linestyle=grid_linestyle)

        # Correlation ticks on outer arc
        theta_ticks = np.arccos(corr_ticks)
        ax.set_xticks(theta_ticks)
        ax.set_xticklabels([f"{c:.2g}" for c in corr_ticks], fontsize=fs_ticks, color="0.25")

        # Radial ticks
        ax.set_yticks(r_ticks)
        ax.set_yticklabels([f"{t:.1f}" for t in r_ticks], fontsize=fs_ticks, color="0.25")

        # Radial axis label
        ax.set_xlabel(
            r"Normalized Standard Deviation ($\sigma_m / \sigma_o$)",
            fontsize=fs_axis_label,
            labelpad=12,
            color="0.2",
        )

        # Subtle correlation label along the arc
        ax.text(
            corr_label_angle,
            r_max * corr_label_offset,
            "Pattern Correlation",
            fontsize=fs_corr_arc,
            ha="center",
            va="bottom",
            rotation=corr_label_rotation,
            color="0.3",
        )

        # Contours of constant centered RMS error (CRMSE)
        rs = np.linspace(0, r_max, 150)
        thetas = np.linspace(0, np.deg2rad(thetamax), 150)
        R_grid, TH_grid = np.meshgrid(rs, thetas)
        X_grid = R_grid * np.cos(TH_grid)
        Y_grid = R_grid * np.sin(TH_grid)
        E_grid = np.sqrt((X_grid - 1.0) ** 2 + Y_grid ** 2)
        cs = ax.contour(
            TH_grid,
            R_grid,
            E_grid,
            levels=crmse_levels,
            colors=crmse_contour_color,
            linestyles=crmse_contour_style,
            linewidths=crmse_contour_width,
            alpha=crmse_contour_alpha,
        )
        ax.clabel(cs, inline=True, fontsize=fs_crmse_clabel, fmt="%.2f", colors=crmse_clabel_color)

        # Reference circle at normalized standard deviation = 1.0
        ax.plot(
            th_ref,
            np.ones_like(th_ref),
            color=ref_circle_color,
            linestyle=ref_circle_style,
            linewidth=ref_circle_width,
            alpha=ref_circle_alpha,
        )

        # Reference point at (theta=0, r=1.0) - modest gold star
        ref_line, = ax.plot(
            0,
            1.0,
            marker=ref_marker,
            markersize=ref_marker_size,
            color=ref_marker_color,
            markeredgecolor=ref_marker_edgecolor,
            markeredgewidth=ref_marker_edgewidth,
            linestyle="none",
            zorder=6,
        )
        if idx == 0:
            legend_handles.append(ref_line)
            legend_labels.append("Reference (Obs)")

        # Plot model trajectories
        for system in metrics_ds.system.values:
            sys_str = str(system)
            if sys_str not in METHOD_STYLES:
                continue
            style = METHOD_STYLES[sys_str]
            sub = metrics_ds.sel(variable=variable, system=sys_str, init_month=init_month)
            r_vals = sub.amplitude_ratio.values
            p_vals = sub.pattern_correlation.values
            leads = sub.L.values

            valid = (
                np.isfinite(r_vals)
                & np.isfinite(p_vals)
                & (p_vals >= (-1.0 if thetamax == 180 else 0.0))
                & (p_vals <= 1.0)
            )
            if not np.any(valid):
                continue

            thetas_mod = np.arccos(p_vals[valid])
            rs_mod = r_vals[valid]
            leads_valid = leads[valid]
            n_pts = len(leads_valid)

            color = style["color"]
            marker_shape = style["marker"]
            short_name = style["short_name"]

            # Light connecting line
            if connect_leads:
                ax.plot(
                    thetas_mod,
                    rs_mod,
                    color=color,
                    linestyle="-",
                    linewidth=traj_linewidth,
                    alpha=traj_line_alpha,
                    zorder=4,
                )

            # Progressive marker size and alpha across lead time
            s_min, s_max = marker_size_range
            a_min, a_max = marker_alpha_range
            sizes = np.linspace(s_min, s_max, n_pts)
            alphas = np.linspace(a_min, a_max, n_pts)

            for k, (th, r, l, sz, al) in enumerate(zip(thetas_mod, rs_mod, leads_valid, sizes, alphas)):
                lead_label = display_lead(l)
                ax.plot(
                    th,
                    r,
                    marker=marker_shape,
                    markersize=sz,
                    color=color,
                    alpha=al,
                    markeredgecolor="black" if (k == 0 or k == n_pts - 1) else color,
                    markeredgewidth=0.9 if (k == 0 or k == n_pts - 1) else 0.4,
                    zorder=5,
                )
                # Annotate only first and last lead
                if k == 0 or k == n_pts - 1:
                    ax.annotate(
                        f"L{lead_label}",
                        xy=(th, r),
                        xytext=annot_xytext,
                        textcoords="offset points",
                        fontsize=fs_lead_annot,
                        ha="center",
                        va="bottom",
                        color=color,
                        fontweight="bold",
                        bbox=dict(boxstyle=annot_boxstyle, facecolor=annot_facecolor, edgecolor="none", alpha=annot_alpha),
                        zorder=7,
                    )

            if idx == 0:
                h_line, = ax.plot(
                    [], [],
                    marker=marker_shape,
                    markersize=8,
                    color=color,
                    linestyle="-",
                    linewidth=traj_linewidth,
                    markeredgecolor="black",
                    markeredgewidth=0.8,
                )
                legend_handles.append(h_line)
                legend_labels.append(short_name)

        ax.set_title(
            f"{initialization_label(init_month)} initialization",
            fontsize=fs_panel_title,
            fontweight="bold",
            pad=panel_title_pad,
        )

    fig.suptitle(
        f"{index_display}–{variable} Teleconnection Fidelity",
        fontsize=fs_suptitle,
        fontweight="bold",
        y=suptitle_y,
    )

    fig.legend(
        legend_handles,
        legend_labels,
        loc=legend_loc,
        ncol=len(legend_handles),
        fontsize=fs_legend,
        frameon=True,
        framealpha=legend_framealpha,
        edgecolor=legend_edgecolor,
        bbox_to_anchor=legend_bbox,
    )

    plt.subplots_adjust(bottom=adjust_bottom, top=adjust_top, wspace=adjust_wspace)
    path = FIGURE_DIR / (
        f"teleconnection_{index_name.replace('.', '')}_{variable}_"
        f"taylor_diagram.png"
    )
    fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


## 8. Cleanup

Close the notebook cluster and release tracked resources.


In [ ]:
close_notebook_resources(globals())
print("Closed teleconnection-workflow Dask resources.")
